# M05 — PCA reconstruction versus ICA source assumptions

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PM02](../../curriculum/papers/modeling.md#pm02).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** Does a compressed representation preserve the relationships needed by the scientific question?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Dimension reduction replaces a large measurement vector with a smaller set of coordinates. That can reduce noise and computation, but the definition of a useful coordinate depends on the objective. A direction that explains much measurement variance need not predict the scientific target. The basic PCA mechanics appear in the statistics strand; this notebook concentrates on interpretation, reconstruction, and comparison with independent component analysis.

PCA finds orthogonal directions that successively capture variance after centering. With all components and the stored mean, the transform can reconstruct the centered data. Keeping fewer components discards the orthogonal residual. Standardizing before PCA changes which features contribute most variance and therefore changes the representation. In a predictive workflow, centering, scaling, and components must be learned only from training data.

ICA instead models observations as mixtures of sources and seeks components with statistical independence under specified assumptions. It can separate non-Gaussian sources that are mixed linearly, subject to identifiability conditions. Component order, sign, and scale are ambiguous; comparing component number one from two fits as if it were a fixed anatomical object is unsafe. Our evaluation matches components using absolute correlations rather than assuming an order.

The lab mixes two independent non-Gaussian sources into two measurements and adds small noise. A one-component PCA reconstruction loses information. A two-component FastICA fit can recover the synthetic sources up to its ambiguities. This is a transparent mixture model, not a demonstration that every real fMRI artifact or network is an independent source. Spatial ICA and temporal ICA arrange samples and features differently, so the orientation of the input matrix is part of the method.

A component can reflect motion, physiological variation, task structure, or a combination. Labeling a component as noise requires evidence from spatial patterns, spectra, time courses, and acquisition context. Blindly discarding a component because its number resembles an example in a tutorial is indefensible. In real data, component classification and denoising choices require a documented workflow and review.

The failure experiment adds a large variance nuisance dimension and shows that PCA prioritizes it. This is not a failure of the PCA algorithm: variance is exactly its objective. It is a mismatch between that objective and the scientific question. Ask AI to distinguish numerical success, reconstruction quality, predictive usefulness, and biological interpretation, because those are four different claims.

## Transformation contract

Mixed measurements → fitted centering and components → latent coordinates → reconstruction. One-component PCA loses residual variance; ICA preserves an estimated mixing relation but cannot identify source labels, order, sign, or scale uniquely.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from sklearn.decomposition import PCA,FastICA
from scipy.optimize import linear_sum_assignment
rng=np.random.default_rng(405)
S=np.column_stack([rng.laplace(size=1800),rng.uniform(-2,2,1800)])
A=np.array([[1,.7],[.4,1.3]]);X=S@A.T+rng.normal(0,.01,S.shape)
pca=PCA(n_components=1).fit(X[:1200])
reconstruction=pca.inverse_transform(pca.transform(X[1200:]))
error=np.mean((reconstruction-X[1200:])**2)
ica=FastICA(n_components=2,random_state=4,max_iter=2000,tol=1e-5).fit(X[:1200])
recovered=ica.transform(X[1200:])
correlations=np.abs(np.corrcoef(S[1200:].T,recovered.T)[:2,2:])
a,b=linear_sum_assignment(-correlations)
print('PCA reconstruction MSE / matched ICA correlations:',error,correlations[a,b])
assert error>0 and np.min(correlations[a,b])>.95


PCA reconstruction MSE / matched ICA correlations: 0.28667103829033797 [0.99919173 0.99978828]


In [2]:
signal=rng.normal(size=300);nuisance=20*rng.normal(size=300)
X_bad=np.column_stack([signal,nuisance])
leading=PCA(1).fit_transform(X_bad).ravel()
print('Leading PC correlation with signal/nuisance:',np.corrcoef(leading,signal)[0,1],np.corrcoef(leading,nuisance)[0,1])
assert abs(np.corrcoef(leading,nuisance)[0,1])>.99
assert abs(np.corrcoef(leading,signal)[0,1])<.2


Leading PC correlation with signal/nuisance: -0.022489550475652313 0.9999999982919112


## Deliberate failure and repair

PCA follows nuisance variance because that is what we asked it to maximize. Repair begins with the measurement and research question, not merely deleting whichever component is largest. If ICA component signs flip across fits, align components for comparison rather than treating the sign change as reversed biology.

## Your investigation

Explain the shape of samples and features for spatial versus temporal ICA. Plot an original measurement and its one-component reconstruction. Design a training-only comparison of prediction from all features and from a reduced representation. State what evidence would justify labeling a real component as motion-related.

## Transfer to real neuroimaging

Use BrainIAK’s actual fMRI reduction exercise after its data setup. For fMRI ICA denoising, study a validated spatial-ICA workflow and inspect components; the two-channel mixture here is not a replacement for MELODIC or ICA-based artifact classification.

**Primary teaching sources, pinned where hosted on GitHub:**

- [BrainIAK: PCA and feature-selection pipelines](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/04-dimensionality-reduction.ipynb)
- [scikit-learn component decomposition](https://scikit-learn.org/stable/modules/decomposition.html)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Is high explained variance equivalent to task relevance? **No.**
2. Why match ICA components before comparing fits? **Their order, sign, and scale are not uniquely identified.**

### Return to the research question

Reopen [PM02](../../curriculum/papers/modeling.md#pm02) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
